This example requires the following dependencies to be installed:
pip install "lightly[timm]"

In [ ]:
!pip install "lightly[timm]"

Note: The model and training settings do not follow the reference settings
from the paper. The settings are chosen such that the example can easily be
run on a small dataset with a single GPU.

In [ ]:
from __future__ import annotations

In [ ]:
import copy
from functools import partial

In [ ]:
import torch
import torchvision
from timm.models.vision_transformer import vit_small_patch16_224
from torch import Tensor
from torch.nn import Module
from torch.optim import AdamW

In [ ]:
from lightly.loss import FrancaDINOLoss, FrancaIBOTPatchLoss, KoLeoLoss
from lightly.models.modules import FrancaProjectionHead, MaskedVisionTransformerTIMM
from lightly.models.utils import (
    random_cyclic_block_mask,
    update_drop_path_rate,
    update_momentum,
)
from lightly.transforms.dino_transform import DINOTransform
from lightly.utils.scheduler import cosine_schedule, linear_warmup_schedule

In [ ]:
# Franca applies clustering at several nested prefixes of the embedding at once.
NESTING_DIMS = [96, 192, 384]
OUTPUT_DIM = 8192

In [ ]:
def freeze_eval_module(module: Module) -> None:
    """Freeze the parameters of a module."""
    for param in module.parameters():
        param.requires_grad = False
    module.eval()

In [ ]:
def to_view_major(
    head_out: tuple[Tensor, ...], n_views: int
) -> list[tuple[Tensor, ...]]:
    """Reorders a head output from level-major to view-major.

    The head returns one tensor per nesting level, each holding ``n_views`` stacked
    views. FrancaDINOLoss expects one entry per view, each a tuple over the levels.
    """
    chunked = [level.chunk(n_views) for level in head_out]
    return [tuple(level[view] for level in chunked) for view in range(n_views)]

In [ ]:
class FrancaHead(Module):
    def __init__(
        self, dino_head: FrancaProjectionHead, ibot_head: FrancaProjectionHead
    ) -> None:
        super().__init__()
        self.dino_head = dino_head
        self.ibot_head = ibot_head

In [ ]:
class Franca(Module):
    def __init__(
        self,
        ibot_separate_head: bool = False,
    ) -> None:
        super().__init__()

        # Backbones
        vit_teacher = vit_small_patch16_224(
            pos_embed="learn",
            dynamic_img_size=True,
            init_values=1e-5,
            pretrained=False,
        )
        self.teacher_backbone = MaskedVisionTransformerTIMM(
            vit=vit_teacher,
            antialias=False,
            pos_embed_initialization="skip",
        )
        self.student_backbone = copy.deepcopy(self.teacher_backbone)
        update_drop_path_rate(
            self.student_backbone.vit,
            drop_path_rate=0.1,
            mode="uniform",
        )

        freeze_eval_module(self.teacher_backbone)

        # Heads
        franca_head = partial(
            FrancaProjectionHead,
            input_dim=384,
            nesting_dims=NESTING_DIMS,
            output_dim=OUTPUT_DIM,
        )

        teacher_dino_head = franca_head()
        student_dino_head = franca_head()

        if ibot_separate_head:
            teacher_ibot_head = franca_head()
            student_ibot_head = franca_head()
        else:
            teacher_ibot_head = teacher_dino_head
            student_ibot_head = student_dino_head

        self.teacher_head = FrancaHead(
            dino_head=teacher_dino_head,
            ibot_head=teacher_ibot_head,
        )
        self.student_head = FrancaHead(
            dino_head=student_dino_head,
            ibot_head=student_ibot_head,
        )

        freeze_eval_module(self.teacher_head)

    def forward(self, x: Tensor) -> Tensor:
        return self.teacher_backbone(x)

    def forward_teacher(self, x: Tensor) -> tuple[Tensor, Tensor]:
        features = self.teacher_backbone.encode(x)
        cls_tokens = features[:, 0]
        return cls_tokens, features

    def forward_student(
        self, x: Tensor, mask: Tensor | None
    ) -> tuple[Tensor, Tensor | None]:
        features = self.student_backbone.encode(x, mask=mask)
        cls_tokens = features[:, 0]
        masked_features = None if mask is None else features[mask]
        return cls_tokens, masked_features

In [ ]:
model = Franca()

In [ ]:
transform = DINOTransform(
    global_crop_scale=(0.32, 1),
    local_crop_scale=(0.05, 0.32),
    n_local_views=8,
)

In [ ]:
# We ignore object detection annotations by setting target_transform to return 0.
def target_transform(t):
    return 0

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
model.to(device)

In [ ]:
dataset = torchvision.datasets.VOCDetection(
    "datasets/pascal_voc",
    download=True,
    transform=transform,
    target_transform=target_transform,
)
# Or create a dataset from a folder containing images or videos.
# dataset = LightlyDataset("path/to/folder")

In [ ]:
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    drop_last=True,
    num_workers=8,
)

In [ ]:
# Create the loss functions. The nested prototype counts come from the head.
dino_criterion = FrancaDINOLoss(output_dims=model.student_head.dino_head.output_dims)
ibot_criterion = FrancaIBOTPatchLoss(
    output_dims=model.student_head.ibot_head.output_dims
)
koleo_criterion = KoLeoLoss()

In [ ]:
# Move loss to correct device because it also contains parameters.
dino_criterion = dino_criterion.to(device)
ibot_criterion = ibot_criterion.to(device)
koleo_criterion = koleo_criterion.to(device)

In [ ]:
optimizer = AdamW(model.parameters(), lr=0.001)

In [ ]:
epochs = 50
num_batches = len(dataloader)
total_steps = epochs * num_batches

In [ ]:
print("Starting Training")
for epoch in range(epochs):
    total_loss = 0
    for batch_idx, batch in enumerate(dataloader):
        views = batch[0]
        views = [view.to(device) for view in views]
        global_views = torch.cat(views[:2])
        local_views = torch.cat(views[2:])

        # Masking
        B = len(global_views)
        sequence_length = model.teacher_backbone.sequence_length
        mask = global_views.new_zeros((B, sequence_length), dtype=torch.bool)

        # Mask patches except class token.
        H, W = model.teacher_backbone.vit.patch_embed.grid_size
        assert H * W == sequence_length - 1, (
            f"Unexpected grid size: {H}x{W}, sequence_length {sequence_length}"
        )
        block_mask = random_cyclic_block_mask(size=(B, H, W), device=mask.device)
        mask[:, 1:] = block_mask.flatten(start_dim=1)

        # Teacher forward
        with torch.no_grad():
            teacher_cls_token, teacher_features = model.forward_teacher(global_views)
            teacher_cls_out = model.teacher_head.dino_head.forward(teacher_cls_token)
            teacher_masked_out = model.teacher_head.ibot_head.forward(
                teacher_features[mask]
            )

        # Student forward
        (
            student_global_cls_token,
            student_global_masked_features,
        ) = model.forward_student(global_views, mask=mask)
        student_global_cls_out = model.student_head.dino_head.forward(
            student_global_cls_token
        )
        student_global_masked_out = model.student_head.ibot_head.forward(
            student_global_masked_features
        )
        student_local_cls_token, _ = model.forward_student(local_views, mask=None)
        student_local_cls_out = model.student_head.dino_head.forward(
            student_local_cls_token
        )
        # Concatenate the global and local student outputs per nesting level.
        student_cls_out = tuple(
            torch.cat([global_out, local_out])
            for global_out, local_out in zip(
                student_global_cls_out, student_local_cls_out
            )
        )

        # Calculate current global step based on epoch and batch index.
        global_step = epoch * num_batches + batch_idx

        # Calculate the loss.
        teacher_temp = linear_warmup_schedule(
            step=global_step,
            warmup_steps=int(30 / epochs * total_steps),
            start_value=0.04,
            end_value=0.07,
        )
        dino_loss = dino_criterion(
            teacher_out=to_view_major(teacher_cls_out, n_views=2),
            student_out=to_view_major(student_cls_out, n_views=len(views)),
            teacher_temp=teacher_temp,
        )
        ibot_loss = ibot_criterion(
            teacher_out=teacher_masked_out,
            student_out=student_global_masked_out,
            mask=block_mask,
            teacher_temp=teacher_temp,
        )
        koleo_loss = 0.1 * sum(
            koleo_criterion(t) for t in student_global_cls_token.chunk(2)
        )
        loss = dino_loss + ibot_loss + koleo_loss

        total_loss += loss.detach()
        loss.backward()

        # Optionally zero out the learning rate of the last layer.
        if epoch < 1:
            for param_group in optimizer.param_groups:
                if "last_layer" in param_group:
                    param_group["lr"] = 0.0

        # Apply weight decay schedule.
        weight_decay = cosine_schedule(
            step=global_step,
            max_steps=total_steps,
            start_value=0.04,
            end_value=0.4,
        )

        # Update weight decay directly for all parameter groups.
        for group in optimizer.param_groups:
            if group["weight_decay"] != 0.0:
                group["weight_decay"] = weight_decay

        optimizer.step()
        optimizer.zero_grad()

        # Momentum update teacher.
        momentum = cosine_schedule(
            step=global_step,
            max_steps=total_steps,
            start_value=0.992,
            end_value=1.0,
        )
        update_momentum(model.student_backbone, model.teacher_backbone, m=momentum)
        update_momentum(model.student_head, model.teacher_head, m=momentum)

    avg_loss = total_loss / len(dataloader)
    print(f"epoch: {epoch:>02}, loss: {avg_loss:.5f}")